In [11]:
from functools import partial

import astroplan as ap
import astropy.units as u

from astropaul.database import database_connection, html_path
import astropaul.targetlistcreator as tlc
import astropaul.html as html
import astropaul.phase as ph
import astropaul.priority as pr

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
name = "APO Observing 2026-Q2"
html_dir = html_path() / name
html.clear_directory(html_dir)

session = tlc.ObservingSession(ap.Observer.at_site("APO"))

session.add_full_day("2026-05-30")
session.add_half_day("2026-05-31", first_half=True)
session.add_full_day("2026-06-01")
session.add_full_day("2026-06-02")

synthetic_phase_percent = 0.02

phase_event_defs = [
    ph.PhaseEventDef("Mid Eclipse", ph.calc_mid_eclipse),
    ph.PhaseEventDef("Eclipse", ph.calc_mid_eclipse),
    ph.PhaseEventDef(
        "Not in Eclipse", partial(ph.calc_time_of_gress, synthetic_phase_percent=synthetic_phase_percent, ingress=False)
    ),
    ph.PhaseEventDef("Eclipse", partial(ph.calc_time_of_gress, synthetic_phase_percent=synthetic_phase_percent, ingress=True)),
]

min_altitude = 35 * u.deg
max_magnitude = 13

with database_connection() as conn:
    creator = tlc.TargetListCreator(name=name, connection=conn, phase_event_defs=phase_event_defs)
    creator.steps = [
        tlc.add_targets,
        tlc.add_lists,
        partial(tlc.filter_targets, criteria=lambda df: df["List Kostov EBs"]),
        tlc.ancillary_data_from_tess,
        partial(tlc.filter_targets, criteria=lambda df: df["Vmag"] < max_magnitude),
        partial(
            tlc.add_observability,
            observing_session=session,
            calc_moon_distance=True,
            observability_threshold=(min_altitude, 80 * u.deg),
        ),
        partial(tlc.filter_targets, criteria=lambda df: df["Observable Any Night"]),
        partial(tlc.add_database_table, table_name="Ephemerides"),
        partial(tlc.add_database_table, table_name="DSSI Observations"),
        # partial(
        #     tlc.add_system_configuration,
        #     table_name="DSSI Observations",
        #     time_column="Mid JD",
        #     eclipse_table="SIDE Observations",
        # ),
        partial(tlc.add_database_table, table_name="Speckle Detections"),
        partial(
            tlc.add_phase_events,
            observing_session=session,
            phase_event_defs=phase_event_defs,
            event_types=["Mid Eclipse", "Eclipse"],
        ),
        partial(tlc.add_database_table, table_name="PEPSI Observations"),
        tlc.add_pepsi_evaluations,
    ]
    tl = creator.calculate(verbose=False)

print(tl.summarize())

Name: APO Observing 2026-Q2
Criteria:
  All targets loaded (732 targets remain)
  lambda df: df["List Kostov EBs"] (198 targets remain)
  lambda df: df["Vmag"] < 13 (111 targets remain)
  Observability calculated at APO in 15.0 min intervals from 2026-05-30 to 2026-06-02
    AltitudeConstraint: {'min': np.float64(35.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: df["Observable Any Night"] (34 targets remain)
  
34 targets:
    34 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    List: (0, 19)
    TESS Data: (4, 0)
    Observable: (5, 16)
    Count: (4, 0)
Associated tables:
    1166 rows,  2 columns: List Memberships
     733 rows, 126 columns: TESS
       4 rows,  2 columns: Lunar Phases
     114 rows,  7 columns: Ephemerides
     134 rows,  9 columns: DSSI Observations
     141 rows, 12 columns: Speckle Detections
     233 rows, 10 columns: Phase Events
     164 rows, 15 columns: PEPSI Observations



In [20]:
html.clear_directory(html_dir)

altitude_categories = [
    ((-90, min_altitude.value), 0),
    ((min_altitude.value, 90), 1),
]

pl = pr.PriorityList(tl, session, interval=30 * u.min)
pr.calculate_altitude_priority(pl, altitude_categories=altitude_categories)
pr.prioritize_phase_sequence(pl, ["Eclipse"], "Eclipse", True, True, True)
# pr.prioritize_side_observation(pl, side_state="Eclipse")
pr.calculate_overall_priority(pl)
pr.aggregate_target_priorities(pl, skip_column_threshold=0.01)
pl.categorize_priorities(bins=[0.00, 0.20, 0.40, 0.6, 1.00], labels=["", "*", "* *", "* * *"])

html.render_observing_pages(tl, pl, {}, html_dir)
# pl.categorical_priorities[0]

/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in ITableOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in ITableOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_und

In [15]:
html.render_observing_pages(tl, pl, {}, html_dir)

/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in ITableOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/opt/anaconda3/envs/Astro/lib/python3.10/site-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in ITableOptions: {'sort'}. You can silence this warning by setting `itables.options.warn_on_und